# Train MLP on CIC-UNSW-NB15

## Setup Instructions

### Option A: Use Google Drive (recommended)
1. Upload `train_mlp.ipynb` to Google Colab
2. Upload the `data/processed/split/` folder to your Google Drive at `Colab Notebooks/anomaly-data/`
3. Run all cells

### Option B: Upload files directly to Colab runtime
1. Upload `data/processed/split/` files manually when prompted

### Files needed:
- `X_train_scale.npy` (313,539 × 76)
- `X_val_scale.npy` (67,188 × 76)
- `X_test_scale.npy` (67,188 × 76)
- `y_train.npy`
- `y_val.npy`
- `y_test.npy`

In [ ]:
# @title Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set paths
DRIVE_DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/anomaly-data'
DRIVE_OUT_DIR = '/content/drive/MyDrive/Colab Notebooks/anomaly-models'
import os
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import json
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Load from Google Drive
X_train = np.load(f'{DRIVE_DATA_DIR}/X_train_scale.npy')
X_val = np.load(f'{DRIVE_DATA_DIR}/X_val_scale.npy')
X_test = np.load(f'{DRIVE_DATA_DIR}/X_test_scale.npy')
y_train = np.load(f'{DRIVE_DATA_DIR}/y_train.npy').flatten()
y_val = np.load(f'{DRIVE_DATA_DIR}/y_val.npy').flatten()
y_test = np.load(f'{DRIVE_DATA_DIR}/y_test.npy').flatten()

n_features = X_train.shape[1]
n_classes = len(np.unique(y_train))

print(f'X_train: {X_train.shape}')
print(f'X_val:   {X_val.shape}')
print(f'X_test:  {X_test.shape}')
print(f'n_features: {n_features}, n_classes: {n_classes}')

In [ ]:
# Compute class weights (inverse frequency)
class_counts = np.bincount(y_train.astype(int))
n_samples = len(y_train)
class_weights = n_samples / (n_classes * class_counts)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

class_names = ['Benign', 'Analysis', 'Backdoor', 'DoS', 'Exploits',
               'Fuzzers', 'Generic', 'Reconnaissance', 'Shellcode', 'Worms']

print('Class weights:')
for i, w in enumerate(class_weights):
    print(f'  {i} ({class_names[i]}): {w:.4f}')

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_features, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.net(x)


model = MLP(n_features, n_classes).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,}')
print(f'Trainable params: {trainable_params:,}')
print(model)

In [ ]:
BATCH_SIZE = 2048
EPOCHS = 200
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long)
)
val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.long)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
train_losses = []
val_losses = []
best_val_loss = float('inf')
best_model_state = None
patience = 15
patience_counter = 0

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_train_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        epoch_train_loss += loss.item() * X_batch.size(0)

    scheduler.step()
    epoch_train_loss /= len(train_dataset)

    model.eval()
    epoch_val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            epoch_val_loss += loss.item() * X_batch.size(0)
    epoch_val_loss /= len(val_dataset)

    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_model_state = model.state_dict().copy()
        patience_counter = 0
    else:
        patience_counter += 1

    if epoch % 10 == 0 or epoch == 1:
        elapsed = time.time() - start_time
        print(f'Epoch {epoch:3d}/{EPOCHS} | '
              f'Train loss: {epoch_train_loss:.4f} | '
              f'Val loss: {epoch_val_loss:.4f} | '
              f'LR: {scheduler.get_last_lr()[0]:.2e} | '
              f'Time: {elapsed:.1f}s')

    if patience_counter >= patience:
        print(f'Early stopping at epoch {epoch}')
        break

total_time = time.time() - start_time
print(f'\nTraining complete in {total_time:.1f}s ({total_time/60:.1f} min)')
print(f'Best val loss: {best_val_loss:.4f}')

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train loss')
plt.plot(val_losses, label='Val loss')
plt.axvline(x=np.argmin(val_losses), color='red', linestyle='--', alpha=0.5, label='Best val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training History')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Load best model
model.load_state_dict(best_model_state)

model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for X_batch, y_batch in val_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

macro_f1 = f1_score(all_labels, all_preds, average='macro')
weighted_f1 = f1_score(all_labels, all_preds, average='weighted')
precision = precision_score(all_labels, all_preds, average='macro')
recall = recall_score(all_labels, all_preds, average='macro')

print('=' * 60)
print('Validation Set Results')
print('=' * 60)
print(f'Macro F1:     {macro_f1:.4f}')
print(f'Weighted F1:  {weighted_f1:.4f}')
print(f'Macro Precision: {precision:.4f}')
print(f'Macro Recall:    {recall:.4f}')
print()
print('Per-class F1:')
print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(12, 10))
plt.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar()
tick_marks = np.arange(n_classes)
plt.xticks(tick_marks, class_names, rotation=45, ha='right')
plt.yticks(tick_marks, class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix (Validation Set)')

thresh = cm.max() / 2
for i in range(n_classes):
    for j in range(n_classes):
        color = 'white' if cm[i, j] > thresh else 'black'
        plt.text(j, i, format(cm[i, j], 'd'),
                ha='center', va='center', color=color, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.long)
)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

all_preds = []
all_labels = []
all_probs = []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

macro_f1 = f1_score(all_labels, all_preds, average='macro')
weighted_f1 = f1_score(all_labels, all_preds, average='weighted')

print('=' * 60)
print('Test Set Results')
print('=' * 60)
print(f'Macro F1:     {macro_f1:.4f}')
print(f'Weighted F1:  {weighted_f1:.4f}')
print()
print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

In [ ]:
save_path = f'{DRIVE_OUT_DIR}/mlp_model.pth'
torch.save({
    'model_state_dict': best_model_state,
    'n_features': n_features,
    'n_classes': n_classes,
    'class_names': class_names,
    'class_weights': class_weights.tolist(),
    'test_macro_f1': float(macro_f1),
    'test_weighted_f1': float(weighted_f1),
}, save_path)
print(f'Model saved to: {save_path}')

# Also save to local runtime (download from Colab files panel)
torch.save({
    'model_state_dict': best_model_state,
    'n_features': n_features,
    'n_classes': n_classes,
    'class_names': class_names,
    'class_weights': class_weights.tolist(),
    'test_macro_f1': float(macro_f1),
    'test_weighted_f1': float(weighted_f1),
}, 'mlp_model.pth')
print('Model also saved locally as mlp_model.pth')

In [ ]:
print('=' * 60)
print('MLP Training Summary')
print('=' * 60)
print(f'Architecture: 512 → 256 → 128 → {n_classes}')
print(f'Total params: {total_params:,}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Optimizer: AdamW (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})')
print(f'Scheduler: CosineAnnealingLR')
print(f'Loss: CrossEntropy (class-weighted)')
print(f'Early stopping: patience={patience}')
print(f'Best val loss: {best_val_loss:.4f}')
print(f'Training time: {total_time:.1f}s ({total_time/60:.1f} min)')
print(f'Test Macro F1: {macro_f1:.4f}')
print(f'Test Weighted F1: {weighted_f1:.4f}')